# Week 7: Post-Class Exercise - SOLUTIONS

**Complete solutions with explanations**

---

## Part 1: Setup and Data Loading

In [ ]:
# Setup
import os
os.environ['KERAS_BACKEND'] = 'torch'

import keras
from keras import layers
from keras.callbacks import EarlyStopping
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

print(f"Keras version: {keras.__version__}")
print("Setup complete!")

In [ ]:
# Load Fashion-MNIST dataset
from keras.datasets import fashion_mnist

(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

print(f"Full training set: {X_train_full.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Normalize pixel values to 0-1 range
X_train_full = X_train_full / 255.0
X_test = X_test / 255.0

print(f"After normalization:")
print(f"Training range: {X_train_full.min():.2f} to {X_train_full.max():.2f}")
print(f"Test range: {X_test.min():.2f} to {X_test.max():.2f}")

In [ ]:
# Three-way split: 80% train, 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Flatten images for Dense layers
X_train_flat = X_train.reshape(-1, 784)
X_val_flat = X_val.reshape(-1, 784)
X_test_flat = X_test.reshape(-1, 784)

print(f"Flattened training: {X_train_flat.shape}")
print(f"Flattened validation: {X_val_flat.shape}")
print(f"Flattened test: {X_test_flat.shape}")

---

## Part 2: Build Model with Dropout

In [ ]:
# Build model with Dropout regularization
model = keras.Sequential([
    layers.Input(shape=(784,)),
    
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),  # Drop 30% of neurons during training
    
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    
    layers.Dense(10, activation='softmax')  # NO Dropout before output
])

print("Model architecture:")
model.summary()

In [ ]:
# Compile model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled!")

---

## Part 3: Configure EarlyStopping

In [ ]:
# Create EarlyStopping callback
early_stop = EarlyStopping(
    monitor='val_loss',        # Watch validation loss
    patience=5,                # Wait 5 epochs before stopping
    restore_best_weights=True, # Restore weights from best epoch
    verbose=1                  # Print when stopping
)

print("EarlyStopping configured!")

---

## Part 4: Train Model

In [ ]:
# Train model with Dropout + EarlyStopping
print("Training model with regularization...")
history = model.fit(
    X_train_flat, y_train,
    validation_data=(X_val_flat, y_val),
    epochs=50,  # Set high - EarlyStopping will stop us
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

print("\nTraining complete!")

---

## Part 5: Plot Training Curves

In [ ]:
# Plot training curves
plt.figure(figsize=(12, 4))

# Loss plot
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend()
plt.title('Loss Over Time')
plt.grid(True, alpha=0.3)

# Accuracy plot
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend()
plt.title('Accuracy Over Time')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final training performance
train_acc = history.history['accuracy'][-1]
val_acc = history.history['val_accuracy'][-1]
print(f"\nFinal Training Accuracy: {train_acc:.4f}")
print(f"Final Validation Accuracy: {val_acc:.4f}")
print(f"Gap: {(train_acc - val_acc):.4f}")

**Analysis:**

The curves should show "Healthy Learning" pattern:
- Both training and validation loss decreasing
- Both accuracies increasing
- Small gap between train/val (< 0.05)
- EarlyStopping likely triggered around epoch 20-30

This indicates Dropout and EarlyStopping successfully prevented overfitting.

---

## Part 6: Final Evaluation on Test Set

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test_flat, y_test, verbose=0)

print(f"\nFinal Test Results:")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

# Compare to validation
print(f"\nValidation Accuracy: {val_acc:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Difference: {abs(val_acc - test_accuracy):.4f}")

if test_accuracy >= 0.85:
    print("\nSUCCESS! Test accuracy >= 85%")
else:
    print("\nTest accuracy < 85%. Consider training longer or adjusting architecture.")

**Expected Results:**
- Test accuracy: 87-89%
- Very close to validation accuracy (within 1-2%)
- This confirms good generalization!

---

## Part 7: Save Model

In [ ]:
# Save complete model
model.save('week7_fashion_mnist.keras')
print("Model saved!")

# Verify loading works
from keras.models import load_model
loaded_model = load_model('week7_fashion_mnist.keras')
print("Model loaded successfully!")

# Test prediction
sample_prediction = loaded_model.predict(X_test_flat[:1], verbose=0)
print(f"\nSample prediction shape: {sample_prediction.shape}")
print(f"Predicted class: {np.argmax(sample_prediction[0])}")
print(f"Actual class: {y_test[0]}")

---

## Part 8: Bonus Visualizations

In [ ]:
# Visualize predictions
class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Get predictions for first 10 test images
predictions = model.predict(X_test_flat[:10], verbose=0)
predicted_classes = np.argmax(predictions, axis=1)

plt.figure(figsize=(15, 3))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_test[i], cmap='gray')
    
    # Color: green if correct, red if wrong
    color = 'green' if predicted_classes[i] == y_test[i] else 'red'
    plt.title(f"Pred: {class_names[predicted_classes[i]]}\nTrue: {class_names[y_test[i]]}",
              color=color, fontsize=9)
    plt.axis('off')

plt.tight_layout()
plt.show()

# Calculate accuracy for these 10
correct = np.sum(predicted_classes == y_test[:10])
print(f"\nCorrect predictions: {correct}/10")

---

## Key Takeaways

**What made this model production-ready:**

1. **Three-way split:** Proper train/val/test separation prevents information leakage

2. **Dropout regularization:** 
   - Added after each Dense layer
   - Rate 0.3 is a good default
   - Forces robust feature learning

3. **EarlyStopping callback:**
   - Monitors val_loss to detect overfitting
   - Patience=5 gives model time to improve
   - restore_best_weights ensures we keep best epoch

4. **Training curves analysis:**
   - Visualize to diagnose problems
   - Look for gap between train/val
   - Healthy learning = curves close together

5. **Model persistence:**
   - .keras format saves everything
   - Easy to deploy to production

**Common Mistakes to Avoid:**
- Forgetting validation_data in fit() → EarlyStopping won't work
- Adding Dropout after output layer → Reduces prediction quality
- Not normalizing data → Training instability
- Touching test set during training → Optimistic performance

---

## Extension Solutions

**1. Architecture Experiment (256→128→64):**

```python
model_larger = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])
```
**Expected:** ~88-90% accuracy, slightly better but more parameters

**2. Dropout Rate Tuning:**
- 0.2: Lighter regularization, may overfit slightly, ~88% accuracy
- 0.3: Good balance (our solution), ~87-89% accuracy
- 0.4: Stronger regularization, ~86-88% accuracy
- 0.5: Very strong, may underfit, ~85-87% accuracy

**3. Patience Experiment:**
- patience=3: Stops earlier (~15-20 epochs), may stop too soon
- patience=5: Good balance (our solution), ~20-30 epochs
- patience=10: Trains longer (~30-40 epochs), risk of slight overfitting

**4. No Regularization Comparison:**
Without Dropout/EarlyStopping:
- Training accuracy: 94-96%
- Validation accuracy: 85-87%
- Large gap (8-10%) = overfitting!
- Test accuracy: 85-87% (worse than with regularization)

**Conclusion:** Regularization improves validation/test accuracy by 2-3% and reduces overfitting.

---

*Week 7 Post-Class Solutions | Version 1.0 | February 2026*